# Atividade — Meios de transporte das mulheres (Brasil, 2022)

**Arquivo utilizado:** `Tabela_13_Meio_de_transporte.xlsx`

Nesta atividade serão analisados os meios de transporte utilizados por mulheres de 10 anos ou mais de idade, ocupadas na semana de referência, segundo cor ou raça.

## PARTE 1 — Gráfico de pizza

In [1]:
%pip install pandas matplotlib seaborn openpyxl

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/80.3 kB ? eta -:--:--
     -------------- ----------------------- 30.7/80.3 kB 640.0 kB/s eta 0:00:01
     ---------------------------------------- 80.3/80.3 kB 1.1 MB/s eta 0:00:00
     ---------------------------------------- 0.0/131.7 kB ? eta -:--:--
     -------------------------------------- 131.7/131.7 kB 2.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.2/9.7 MB 15.7 MB/s eta 0:00:01
   -- ------------------------------------- 0.6/9.7 MB 8.1 MB/s eta 0:00:02
   ----- ---------------------------------- 1.4/9.7 MB 12.7 MB/s eta 0:00:01
   ------------ --------------------------- 2.9/9.7 MB 17.1 MB/s eta 0:00:01
   ----------------- ---------------------- 4.3/9.7 MB 21.3 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.7 MB 27.2 MB/s eta 0:00:01
   ----


[notice] A new release of pip is available: 23.3.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

arquivo = 'Tabela_13_Meio_de_transporte.xlsx'
df = pd.read_excel(arquivo, sheet_name='Exemplo gráfico Brasil', header=0)

df.columns = ['Meio de Transporte', 'Branca', 'Preta ou Parda', 'Indígena']
df = df.iloc[1:].copy()

for col in ['Branca', 'Preta ou Parda', 'Indígena']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

cores = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6']
grupos = ['Branca', 'Preta ou Parda', 'Indígena']

for i, col in enumerate(grupos):
    axes[i].pie(
        df[col],
        labels=df['Meio de Transporte'],
        autopct='%1.3f%%', 
        startangle=180,
        colors=cores,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.2},
    )
    axes[i].set_title(f'Cor/Raça: {col}', fontsize=12, fontweight='bold')

plt.suptitle(
    'Meios de transporte utilizados por mulheres — Brasil (2022)',
    fontsize=14,
    fontweight='bold',
)

plt.tight_layout()
plt.savefig('grafico_transporte.png', dpi=300)
plt.show()

## Análise dos gráficos

Os valores abaixo mostram a **diferença em pontos percentuais (p.p.)** entre as categorias de cor/raça para cada meio de transporte. Isso é mais adequado do que dizer apenas “%”, porque estamos comparando percentuais.

- **A pé:** Branca × Preta ou Parda = **5.4 p.p.**; Branca × Indígena = **18.1 p.p.**; Preta ou Parda × Indígena = **12.7 p.p.**
- **Bicicleta:** Branca × Preta ou Parda = **2.0 p.p.**; Branca × Indígena = **2.7 p.p.**; Preta ou Parda × Indígena = **0.7 p.p.**
- **Motocicleta ou Mototaxi:** Branca × Preta ou Parda = **4.3 p.p.**; Branca × Indígena = **5.5 p.p.**; Preta ou Parda × Indígena = **1.2 p.p.**
- **Automóvel, taxi ou assemelhados:** Branca × Preta ou Parda = **21.2 p.p.**; Branca × Indígena = **26.6 p.p.**; Preta ou Parda × Indígena = **5.4 p.p.**
- **Transporte coletivo:** Branca × Preta ou Parda = **9.4 p.p.**; Branca × Indígena = **2.8 p.p.**; Preta ou Parda × Indígena = **12.2 p.p.**
- **Outros:** Branca × Preta ou Parda = **0.1 p.p.**; Branca × Indígena = **3.1 p.p.**; Preta ou Parda × Indígena = **3.0 p.p.**

### Resumo

A maior diferença entre Branca e Preta ou Parda aparece no **automóvel, táxi ou assemelhados**, com **21,2 p.p.** de diferença. Entre Branca e Indígena, a maior diferença também aparece no **automóvel, táxi ou assemelhados**, com **26,6 p.p.**. Entre Preta ou Parda e Indígena, a maior diferença aparece no **transporte coletivo**, com **12,2 p.p.**.


## PARTE 2 — Estados, `melt()` e `groupby()`

In [ ]:
raw = pd.read_excel(arquivo, sheet_name='BR GR UF MU', header=None)

dados = raw.iloc[3:, :19].copy()

racas = ['Branca', 'Preta ou Parda', 'Indígena']
meios = [
    'A pé',
    'Bicicleta',
    'Motocicleta ou Mototaxi',
    'Automóvel, taxi ou assemelhados',
    'Transporte coletivo',
    'Outros'
]

dados.columns = ['Local'] + [f'{r}_{m}' for r in racas for m in meios]
dados = dados.reset_index(drop=True)

for col in dados.columns[1:]:
    dados[col] = pd.to_numeric(dados[col], errors='coerce')

dados.head()

In [ ]:
estados = dados.iloc[6:33].copy()

print('Estados encontrados:')
print(estados['Local'].to_string(index=False))

In [ ]:
colunas_melt = []

for raca in racas:
    for meio in meios:
        colunas_melt.append(f'{raca}_{meio}')

estados_melt = estados.melt(
    id_vars='Local',
    value_vars=colunas_melt,
    var_name='Categoria',
    value_name='Percentual'
)

estados_melt[['Raça', 'Meio de Transporte']] = estados_melt['Categoria'].str.split('_', n=1, expand=True)

estados_melt = estados_melt.drop(columns='Categoria')

estados_melt.head()

In [ ]:

estados_geral = (
    estados_melt
    .groupby(['Local', 'Meio de Transporte'], as_index=False)['Percentual']
    .mean()
)

coletivo_estados = (
    estados_geral[estados_geral['Meio de Transporte'] == 'Transporte coletivo']
    .sort_values('Percentual', ascending=False)
)

print('Estado com MAIS transporte coletivo:')
print(coletivo_estados.iloc[0])

print('\nEstado com MENOS transporte coletivo:')
print(coletivo_estados.iloc[-1])

In [ ]:
municipios = dados.iloc[33:].copy()

municipios_melt = municipios.melt(
    id_vars='Local',
    value_vars=colunas_melt,
    var_name='Categoria',
    value_name='Percentual'
)

municipios_melt[['Raça', 'Meio de Transporte']] = municipios_melt['Categoria'].str.split('_', n=1, expand=True)
municipios_melt = municipios_melt.drop(columns='Categoria')

municipios_geral = (
    municipios_melt
    .groupby(['Local', 'Meio de Transporte'], as_index=False)['Percentual']
    .mean()
)

bicicleta = (
    municipios_geral[municipios_geral['Meio de Transporte'] == 'Bicicleta']
    .sort_values('Percentual', ascending=False)
)

carro = (
    municipios_geral[municipios_geral['Meio de Transporte'] == 'Automóvel, taxi ou assemelhados']
    .sort_values('Percentual', ascending=False)
)

print('Cidade com MAIS mulheres utilizando bicicleta:')
print(bicicleta.iloc[0])

print('\nCidade com MAIS mulheres utilizando carro:')
print(carro.iloc[0])

## Respostas

> **Observação:** como a planilha apresenta percentuais separados por cor/raça e não apresenta a quantidade absoluta de mulheres em cada grupo, “em geral” foi calculado pela **média dos três percentuais** (Branca, Preta ou Parda e Indígena).

1. **Qual estado tem mais mulheres, em geral, utilizando o transporte coletivo?**  
   **Rio de Janeiro — 52.23%**.

2. **Qual estado tem menos mulheres, em geral, utilizando o transporte coletivo?**  
   **Rondônia — 4.33%**.

3. **Qual cidade do Brasil tem mais mulheres andando de bicicleta?**  
   **Tuiuti (SP) — 100.00%**.

4. **Qual cidade do Brasil tem mais mulheres utilizando carro?**  
   **Bom Jesus do Oeste (SC) — 86.05%**.


## Conclusão

A análise mostra diferenças nos meios de transporte conforme a cor ou raça e também entre estados e municípios. Os gráficos da primeira parte permitem visualizar a distribuição dos meios de transporte no Brasil, enquanto o `melt()` reorganiza os dados para facilitar a análise e o `groupby()` permite calcular as médias e encontrar os maiores e menores valores.